<a href="https://colab.research.google.com/github/Saibhossain/face-generation-model/blob/main/DCGAN_with_Data_Augumentation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Data set preparation

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import random
from PIL import Image
from tqdm import tqdm
import torchvision.transforms as transforms

# --- CONFIGURATION ---
INPUT_DIR = "/content/drive/MyDrive/Datasets/Celebrity faces"  # Your original 512 images
OUTPUT_DIR = "/content/drive/MyDrive/Datasets/Augmented_Dataset" # Where to save the 5,000+ images
MULTIPLIER = 12  # How many new images to create per original image

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Define the augmentations
# We use a randomized pipeline to ensure every generated image is slightly different
augmentor = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15), # Rotate +/- 15 degrees
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)), # Shift image slightly
    # Optional: Add Gaussian Blur or Noise if you have it defined
])

def augment_and_save():
    print(f"Reading from: {INPUT_DIR}")
    print(f"Target size: {MULTIPLIER}x original count")

    valid_extensions = ('.png', '.jpg', '.jpeg', '.bmp')
    files = []

    # Recursively find images
    for root, _, filenames in os.walk(INPUT_DIR):
        for filename in filenames:
            if filename.lower().endswith(valid_extensions):
                files.append(os.path.join(root, filename))

    print(f"Found {len(files)} original images.")

    count = 0
    for file_path in tqdm(files):
        try:
            img = Image.open(file_path).convert("RGB")

            # Save the original first
            original_name = f"orig_{count}.png"
            img.save(os.path.join(OUTPUT_DIR, original_name))

            # Generate variations
            for i in range(MULTIPLIER):
                aug_img = augmentor(img)
                aug_name = f"aug_{count}_{i}.png"
                aug_img.save(os.path.join(OUTPUT_DIR, aug_name))

            count += 1
        except Exception as e:
            print(f"Error processing {file_path}: {e}")

    print(f"\n Done! Dataset expanded to {len(os.listdir(OUTPUT_DIR))} images.")
    print(f"Saved in: {OUTPUT_DIR}")

if __name__ == "__main__":
    augment_and_save()

#DCGAN

In [ ]:
import os
import random
import torch
import torch.nn as nn
import torch.optim as optim
import torch.utils.data
import torchvision.datasets as dset
import torchvision.transforms as transforms
import torchvision.utils as vutils
import numpy as np
import matplotlib.pyplot as plt

# --- CONFIGURATION ---
# POINT TO THE AUGMENTED DATASET (From Step 1)
DATAROOT = "/content/drive/MyDrive/Datasets/Augmented_Dataset"
WORKERS = 2
BATCH_SIZE = 128
IMAGE_SIZE = 64
NC = 3
NZ = 100
NGF = 64
NDF = 64
NUM_EPOCHS = 50  # DCGAN needs many epochs
LR = 0.0002
BETA1 = 0.5
NGPU = 1
OUT_DIR = "./dcgan_results"

os.makedirs(OUT_DIR, exist_ok=True)

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(device)

cuda:0


In [ ]:
# --- CUSTOM DATA LOADER (Handles flat folder structure) ---
class FlatFolderDataset(torch.utils.data.Dataset):
    def __init__(self, root, transform=None):
        self.root = root
        self.files = [os.path.join(root, f) for f in os.listdir(root)
                      if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp'))]
        self.transform = transform

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        from PIL import Image
        try:
            img = Image.open(self.files[idx]).convert('RGB')
            if self.transform:
                img = self.transform(img)
            return img, 0
        except Exception as e:
            print(f"Error loading {self.files[idx]}: {e}")
            return torch.zeros(3, IMAGE_SIZE, IMAGE_SIZE), 0

def get_dataloader():
    dataset = FlatFolderDataset(
        root=DATAROOT,
        transform=transforms.Compose([
            transforms.Resize(IMAGE_SIZE),
            transforms.CenterCrop(IMAGE_SIZE),
            transforms.ToTensor(),
            transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
        ])
    )
    return torch.utils.data.DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=WORKERS)

# --- MODELS ---
def weights_init(m):
    classname = m.__class__.__name__
    if classname.find('Conv') != -1:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif classname.find('BatchNorm') != -1:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)

In [ ]:
class Generator(nn.Module):
    def __init__(self):
        super(Generator, self).__init__()
        self.main = nn.Sequential(
            # Z -> 4x4
            nn.ConvTranspose2d(NZ, NGF * 8, 4, 1, 0, bias=False),
            nn.BatchNorm2d(NGF * 8),
            nn.ReLU(True),
            # 4x4 -> 8x8
            nn.ConvTranspose2d(NGF * 8, NGF * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(NGF * 4),
            nn.ReLU(True),
            # 8x8 -> 16x16
            nn.ConvTranspose2d(NGF * 4, NGF * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(NGF * 2),
            nn.ReLU(True),
            # 16x16 -> 32x32
            nn.ConvTranspose2d(NGF * 2, NGF, 4, 2, 1, bias=False),
            nn.BatchNorm2d(NGF),
            nn.ReLU(True),
            # 32x32 -> 64x64
            nn.ConvTranspose2d(NGF, NC, 4, 2, 1, bias=False),
            nn.Tanh()
        )

    def forward(self, input):
        return self.main(input)

In [ ]:
class Discriminator(nn.Module):
    def __init__(self):
        super(Discriminator, self).__init__()
        self.main = nn.Sequential(
            # 64x64 -> 32x32
            nn.Conv2d(NC, NDF, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            # 32x32 -> 16x16
            nn.Conv2d(NDF, NDF * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(NDF * 2),
            nn.LeakyReLU(0.2, inplace=True),
            # 16x16 -> 8x8
            nn.Conv2d(NDF * 2, NDF * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(NDF * 4),
            nn.LeakyReLU(0.2, inplace=True),
            # 8x8 -> 4x4
            nn.Conv2d(NDF * 4, NDF * 8, 4, 2, 1, bias=False),
            nn.BatchNorm2d(NDF * 8),
            nn.LeakyReLU(0.2, inplace=True),
            # 4x4 -> 1x1
            nn.Conv2d(NDF * 8, 1, 4, 1, 0, bias=False),
            nn.Sigmoid()
        )

    def forward(self, input):
        return self.main(input)

In [ ]:
# ---------------- TRAINING ----------------
def train():
    print("Starting DCGAN Training")

    dataloader = get_dataloader()

    netG = Generator().to(device)
    netD = Discriminator().to(device)

    netG.apply(weights_init)
    netD.apply(weights_init)

    criterion = nn.BCELoss()
    fixed_noise = torch.randn(64, NZ, 1, 1, device=device)

    optimizerD = optim.Adam(netD.parameters(), lr=LR, betas=(BETA1, 0.999))
    optimizerG = optim.Adam(netG.parameters(), lr=LR, betas=(BETA1, 0.999))

    best_g_loss = float("inf")
    BEST_G_PATH = os.path.join(OUT_DIR, "best_generator.pth")
    BEST_D_PATH = os.path.join(OUT_DIR, "best_discriminator.pth")

    for epoch in range(NUM_EPOCHS):
        for i, (data, _) in enumerate(dataloader):

            # ---- Train Discriminator ----
            netD.zero_grad()
            real = data.to(device)
            b_size = real.size(0)
            label = torch.full((b_size,), 1., device=device)

            output = netD(real).view(-1)
            errD_real = criterion(output, label)
            errD_real.backward()

            noise = torch.randn(b_size, NZ, 1, 1, device=device)
            fake = netG(noise)
            label.fill_(0.)

            output = netD(fake.detach()).view(-1)
            errD_fake = criterion(output, label)
            errD_fake.backward()

            optimizerD.step()

            # ---- Train Generator ----
            netG.zero_grad()
            label.fill_(1.)
            output = netD(fake).view(-1)
            errG = criterion(output, label)
            errG.backward()
            optimizerG.step()

            # ---- Save BEST model ----
            if errG.item() < best_g_loss:
                best_g_loss = errG.item()

                torch.save({
                    "epoch": epoch,
                    "generator_state_dict": netG.state_dict(),
                    "optimizerG_state_dict": optimizerG.state_dict(),
                    "g_loss": best_g_loss
                }, BEST_G_PATH)

                torch.save({
                    "epoch": epoch,
                    "discriminator_state_dict": netD.state_dict(),
                    "optimizerD_state_dict": optimizerD.state_dict(),
                    "d_loss": (errD_real + errD_fake).item()
                }, BEST_D_PATH)

                print(f"Best model saved | Epoch {epoch} | G Loss {best_g_loss:.4f}")

            if i % 50 == 0:
                print(
                    f"[{epoch}/{NUM_EPOCHS}] "
                    f"[{i}/{len(dataloader)}] "
                    f"Loss_D: {(errD_real+errD_fake).item():.4f} "
                    f"Loss_G: {errG.item():.4f}"
                )

        with torch.no_grad():
            fake = netG(fixed_noise).detach().cpu()
        vutils.save_image(
            fake,
            f"{OUT_DIR}/epoch_{epoch}.png",
            normalize=True
        )

    print("DCGAN Training Finished")


if __name__ == "__main__":
    train()

🚀 Starting DCGAN Training
✅ Best model saved | Epoch 0 | G Loss 3.1452
[0/50] [0/52] Loss_D: 1.8869 Loss_G: 3.1452
[0/50] [50/52] Loss_D: 0.8447 Loss_G: 24.4537
[1/50] [0/52] Loss_D: 0.1682 Loss_G: 10.9143
✅ Best model saved | Epoch 1 | G Loss 3.1376
✅ Best model saved | Epoch 1 | G Loss 2.9556
✅ Best model saved | Epoch 1 | G Loss 2.8702
[1/50] [50/52] Loss_D: 1.0345 Loss_G: 7.9673
[2/50] [0/52] Loss_D: 0.3803 Loss_G: 2.9764
✅ Best model saved | Epoch 2 | G Loss 1.7953
[2/50] [50/52] Loss_D: 0.5143 Loss_G: 5.5525
[3/50] [0/52] Loss_D: 0.5290 Loss_G: 4.3669
✅ Best model saved | Epoch 3 | G Loss 1.6747
[3/50] [50/52] Loss_D: 0.3129 Loss_G: 5.3357
[4/50] [0/52] Loss_D: 0.4873 Loss_G: 5.2558
[4/50] [50/52] Loss_D: 0.4897 Loss_G: 3.6301
[5/50] [0/52] Loss_D: 0.5204 Loss_G: 3.2694
✅ Best model saved | Epoch 5 | G Loss 1.4316
[5/50] [50/52] Loss_D: 0.6357 Loss_G: 5.1909
[6/50] [0/52] Loss_D: 0.6907 Loss_G: 6.3274
[6/50] [50/52] Loss_D: 0.6498 Loss_G: 6.1291
[7/50] [0/52] Loss_D: 0.5347 Loss_